In [1]:
import os

In [2]:
%pwd

'c:\\projects\\Mlops_Project\\research'

In [3]:
os.chdir('../')

In [11]:
%pip install dagshub

  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached mdurl-0.1.2-py3-none-any.whl.metadata (1.6 kB)
  Using cached requests_toolbelt-1.0.0-py2.py3-none-any.whl.metadata (14 kB)
   ---------------------------------------- 0.0/2.0 MB ? eta -:--:--
   ---------- ----------------------------- 0.5/2.0 MB 2.8 MB/s eta 0:00:01
   -------------------- ------------------- 1.0/2.0 MB 3.0 MB/s eta 0:00:01
   ------------------------------- -------- 1.6/2.0 MB 2.7 MB/s eta 0:00:01
   ---------------------------------------- 2.0/2.0 MB 2.8 MB/s  0:00:00
Using cached annotated_types-0.7.0-py3-none-any.whl (13 kB)
Using cached mdurl-0.1.2-py3-none-any.whl (10.0 kB)
   ---------------------------------------- 0.0/14.1 MB ? eta -:--:--
   - -------------------------------------- 0.5/14.1 MB 2.8 MB/s eta 0:00:05
   -- ------------------------------------- 1.0/14.1 MB 3.0 MB/s eta 0:00:05
   ----- ---------------------------------- 1.8/14.1 MB 2.9 MB/s eta 0:00:05
   ----

In [12]:
import dagshub
dagshub.init(repo_owner='nihadachir', repo_name='Mlops_Project', mlflow=True)

import mlflow
with mlflow.start_run():
  mlflow.log_param('parameter name', 'value')
  mlflow.log_metric('metric name', 1)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=88b9bff4-f96c-4080-bfd4-c1b7834b21d5&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=7d84a287da571e80beddb7b807c44a3ac74b586a188d4091e962e62b0c352b1b


[2025-10-30 14:19:49,316: INFO: _client: HTTP Request: POST https://dagshub.com/login/oauth/middleman "HTTP/1.1 200 OK"]


[2025-10-30 14:19:50,398: INFO: _client: HTTP Request: POST https://dagshub.com/login/oauth/access_token "HTTP/1.1 200 OK"]
[2025-10-30 14:19:51,367: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Accessing as nihadachir

[2025-10-30 14:19:51,389: INFO: helpers: Accessing as nihadachir]
[2025-10-30 14:19:52,286: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/repos/nihadachir/Mlops_Project "HTTP/1.1 200 OK"]
[2025-10-30 14:19:53,154: INFO: _client: HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"]


Initialized MLflow to track repo "nihadachir/Mlops_Project"

[2025-10-30 14:19:53,160: INFO: helpers: Initialized MLflow to track repo "nihadachir/Mlops_Project"]


Repository nihadachir/Mlops_Project initialized!

[2025-10-30 14:19:53,165: INFO: helpers: Repository nihadachir/Mlops_Project initialized!]


In [16]:
from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [5]:
from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [17]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        params_filepath=Params_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH
    ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        # Create main artifacts directory
        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        # Create model evaluation directory
        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name=config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/nihadachir/Mlops_Project.mlflow"  # ✅ Correct URI
        )

        return model_evaluation_config

In [7]:
import setuptools, sys
sys.modules['distutils'] = setuptools._distutils

In [8]:
import os
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib

In [20]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        """
        Calculate standard regression evaluation metrics:
        - RMSE
        - MAE
        - R²
        """
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):
        """
        Evaluate model performance on test data and log:
        - Parameters
        - Metrics
        - Trained model
        to MLflow (via DagsHub)
        """

        # Load test data and model
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        # Split into features and target
        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        # ✅ Set the correct MLflow tracking URI
        mlflow.set_tracking_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            # Predict on test data
            predicted_qualities = model.predict(test_x)

            # Compute evaluation metrics
            rmse, mae, r2 = self.eval_metrics(test_y, predicted_qualities)

            # Save metrics locally
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            # Log params and metrics to MLflow
            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            # ✅ Log model to MLflow
            if tracking_url_type_store != "file":
                # Register model in remote MLflow (DagsHub)
                mlflow.sklearn.log_model(
                    model,
                    artifact_path="model",
                    
                )
            else:
                # Local MLflow tracking — skip registry
                mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"✅ Model evaluation completed and logged to MLflow at: {self.config.mlflow_uri}")

In [21]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2025-10-30 14:34:27,290: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-10-30 14:34:27,290: INFO: common: yaml file: params.yaml loaded successfully]
[2025-10-30 14:34:27,290: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-10-30 14:34:27,306: INFO: common: created directory at: artifacts]
[2025-10-30 14:34:27,307: INFO: common: created directory at: artifacts/model_evaluation]
[2025-10-30 14:34:28,455: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]
✅ Model evaluation completed and logged to MLflow at: https://dagshub.com/nihadachir/Mlops_Project.mlflow
